# Bloco 1 — Formulação do Modelo de Otimização

Formulação conceptual e matemática do problema de escalonamento de equipas de Housekeeping.

Este notebook documenta a formulação do modelo ILP implementado em `src/optimizer.py`, distinguindo claramente as restrições activas no modelo final das que foram incorporadas de outra forma ou que ficaram fora do âmbito desta versão.

> **Cenário:** modelo híbrido — colaboradores internos prioritários, recursos externos activados apenas quando necessário para satisfazer a cobertura mínima.

## 1. Índices e conjuntos

- \( c \in C \): colaboradores internos activos
- \( r \in R \): recursos externos disponíveis
- \( d \in D \): dias do horizonte de planeamento
- \( t \in T \): turnos operacionais \(\{T1, T2\}\)
- \( f \in F \): funções \(\{\text{Auxiliar de limpeza, Empregada de andares, Supervisora}\}\)
- \( s \in S \): semanas do horizonte de planeamento

## 2. Parâmetros

**Cobertura**
- \( N^{min}_{d,t,f} \): cobertura mínima obrigatória por slot \((d,t,f)\)
- \( N^{ideal}_{d,t,f} \): cobertura ideal por slot

**Colaboradores internos**
- \( Disp_{c,d,t} \in \{0,1\} \): disponibilidade do colaborador \(c\) no slot \((d,t)\)
- \( Elegivel_{c,f} \in \{0,1\} \): elegibilidade do colaborador \(c\) para a função \(f\), incorporando qualificação formal e hierarquia funcional
- \( Ativo_c \in \{0,1\} \): indica se o colaborador \(c\) está activo
- \( H^{dia}_c \): horas máximas diárias por contrato
- \( W_c \): dias máximos de trabalho por semana por contrato
- \( M_c \): máximo de dias consecutivos por contrato
- \( Pref_{c,t} \in \{0,1,2\} \): score de preferência de turno
- \( FolgaMatch_{c,d} \in \{0,1\} \): indicador de dia preferido para folga
- \( Custo^{fixo}_c \): custo semanal comprometido do colaborador \(c\), independente das alocações

**Recursos externos**
- \( Cost_r \): custo horário do recurso externo \(r\)
- \( Disp_r,\; Qual_r \): disponibilidade e qualificações do recurso \(r\)

**Turnos**
- \( H_t \): duração em horas do turno \(t\)

## 3. Variáveis de decisão

**Colaboradores internos:**

$$
x_{c,d,t,f} =
\begin{cases}
1, & \text{se o colaborador } c \text{ for alocado ao slot } (d,t,f) \\
0, & \text{caso contrário}
\end{cases}
$$

**Recursos externos:**

$$
y_{r,d,t,f} =
\begin{cases}
1, & \text{se o recurso externo } r \text{ for activado no slot } (d,t,f) \\
0, & \text{caso contrário}
\end{cases}
$$

**Variáveis auxiliares (soft constraints):**
- \( \delta_c \geq 0 \): desvio de carga de trabalho do colaborador \(c\) face ao equilíbrio (S3)
- \( u_{d,t,f} \geq 0 \): défice face à cobertura ideal no slot \((d,t,f)\) (S4)

## 4. Hard constraints

As hard constraints garantem a admissibilidade operacional da solução e não podem ser violadas sem tornar o problema inviável.

As restrições H8 e H9, associadas à elegibilidade funcional e à consideração exclusiva de colaboradores activos, são incorporadas a montante no próprio processo de criação das variáveis de decisão, através dos filtros aplicados aos conjuntos e ao parâmetro \(Elegivel\). A restrição H4, associada ao descanso mínimo entre turnos, ficou fora da formulação activa final por exclusão do turno T3 do âmbito do projecto. As três são documentadas abaixo com o respectivo estatuto.

### H1 — Cobertura mínima por slot

A cobertura mínima deve ser garantida em cada slot \((d,t,f)\), pela soma de internos e externos:

$$
\sum_{c \in C} x_{c,d,t,f} + \sum_{r \in R} y_{r,d,t,f} \geq N^{min}_{d,t,f},
\quad \forall d \in D,\; t \in T,\; f \in F
$$

### H2 — Máximo de um turno por colaborador por dia

$$
\sum_{t \in T} \sum_{f \in F} x_{c,d,t,f} \leq 1, \quad \forall c \in C,\; d \in D
$$

### H3 — Respeito pela disponibilidade declarada

Um colaborador só pode ser alocado a um turno se estiver disponível nesse slot:

$$
\sum_{f \in F} x_{c,d,t,f} \leq Disp_{c,d,t}, \quad \forall c \in C,\; d \in D,\; t \in T
$$

### H4 — Descanso mínimo entre turnos

Um colaborador alocado ao turno nocturno não pode ser alocado ao turno da manhã do dia seguinte, garantindo um intervalo mínimo de descanso:

$$
\sum_{f \in F} x_{c,d,T3,f} + \sum_{f \in F} x_{c,d+1,T1,f} \leq 1,
\quad \forall c \in C,\; d \in D
$$

> **Nota:** o modelo final implementado considera apenas os turnos T1 e T2. O turno T3 foi excluído do âmbito do projecto, pelo que H4 não integra a formulação activa. Permanece documentada como restrição conceptualmente válida e relevante para uma extensão futura com turnos nocturnos.

### H5 — Horas diárias ≤ limite contratual

$$
\sum_{t \in T} \sum_{f \in F} H_t \cdot x_{c,d,t,f} \leq H^{dia}_c,
\quad \forall c \in C,\; d \in D
$$

### H6 — Dias por semana ≤ limite contratual

$$
\sum_{d \in D_s} \sum_{t \in T} \sum_{f \in F} x_{c,d,t,f} \leq W_c,
\quad \forall c \in C,\; s \in S
$$

onde \(D_s\) é o conjunto de dias pertencentes à semana \(s\).

### H8 — Elegibilidade funcional por função

Um colaborador só pode ser alocado a uma função se for elegível para ela:

$$
x_{c,d,t,f} \leq Elegivel_{c,f}, \quad \forall c \in C,\; d \in D,\; t \in T,\; f \in F
$$

O parâmetro \(Elegivel_{c,f}\) incorpora a qualificação formal e a hierarquia funcional — uma Auxiliar pode actuar como Empregada, uma Empregada como Supervisora — com exclusões individuais quando aplicável.

> **Nota:** na implementação final, esta condição é aplicada a montante, através dos filtros sobre os conjuntos e o parâmetro \(Elegivel\), não surgindo como restrição explícita no modelo ILP.

### H9 — Apenas colaboradores activos podem ser alocados

$$
\sum_{t \in T} \sum_{f \in F} x_{c,d,t,f} \leq Ativo_c,
\quad \forall c \in C,\; d \in D
$$

onde \(Ativo_c = 1\) se o colaborador \(c\) estiver activo, \(0\) caso contrário.

> **Nota:** na implementação final, esta condição fica embutida na construção do conjunto \(C\), que inclui apenas colaboradores com \(Ativo_c = 1\). H9 não surge como restrição separada no modelo ILP, mas permanece válida como descrição conceptual do modelo.

### H10 — Máximo de dias consecutivos

$$
\sum_{d' = d}^{d + M_c} \sum_{t \in T} \sum_{f \in F} x_{c,d',t,f} \leq M_c,
\quad \forall c \in C,\; d \in D
$$

onde \(M_c\) é o máximo de dias consecutivos permitido para o colaborador \(c\).

## 5. Soft constraints

As soft constraints representam critérios desejáveis de qualidade da solução. Podem ser violadas, mas a violação é penalizada na função objetivo com os pesos \(w_1, w_2, w_3\).

### S1 — Preferência de turno

Penalização proporcional ao afastamento do turno preferido:

$$
Pen^{pref}_{c,d,t,f} = (2 - Pref_{c,t}) \cdot x_{c,d,t,f}
$$

onde \(Pref_{c,t} \in \{0, 1, 2\}\) — turno a evitar, indiferente ou preferido.

### S2 — Folga no dia preferido

Penalização por alocação no dia preferido para folga:

$$
Pen^{folga}_{c,d} = FolgaMatch_{c,d} \cdot \sum_{t \in T}\sum_{f \in F} x_{c,d,t,f}
$$

onde \(FolgaMatch_{c,d} = 1\) se o dia \(d\) corresponder ao dia de folga preferido do colaborador \(c\).

### S3 — Equidade na distribuição da carga

Penalização por desequilíbrio na carga de trabalho entre colaboradores:

$$
\delta_c \geq Carga_c - \bar{Carga}, \quad
\delta_c \geq -(Carga_c - \bar{Carga}), \quad \forall c \in C
$$

onde \(Carga_c = \sum_{d,t,f} H_t \cdot x_{c,d,t,f}\) e \(\bar{Carga}\) é a carga média de referência.

### S4 — Défice face à cobertura ideal

$$
\sum_{c \in C} x_{c,d,t,f} + \sum_{r \in R} y_{r,d,t,f} + u_{d,t,f}
\geq N^{ideal}_{d,t,f}, \quad \forall d \in D,\; t \in T,\; f \in F
$$

onde \(u_{d,t,f} \geq 0\) é o défice face à cobertura ideal.

## 6. Função objetivo

O custo dos colaboradores internos é **fixo e comprometido** — resulta do contrato independentemente das alocações semanais. Por esse motivo, não entra na função objetivo: minimizá-lo não alteraria a decisão de alocação. É reportado nos KPIs como custo fixo semanal.

A função objetivo minimiza o **custo externo incremental** e as penalizações das soft constraints:

$$
\min Z =
\underbrace{\sum_{r \in R}\sum_{d \in D}\sum_{t \in T}\sum_{f \in F}
Cost_r \cdot H_t \cdot y_{r,d,t,f}}_{\text{custo externo incremental}}
+ w_1 \underbrace{\left(
  \sum_{c,d,t,f} Pen^{pref}_{c,d,t,f} + \sum_{c,d} Pen^{folga}_{c,d}
\right)}_{S1 + S2}
+ w_2 \underbrace{\sum_{d,t,f} u_{d,t,f}}_{S4}
+ w_3 \underbrace{\sum_{c \in C} \delta_c}_{S3}
$$

onde \(w_1, w_2, w_3\) são pesos de penalização calibrados por análise de sensibilidade (valores finais: \(w_1 = 5,\; w_2 = 10,\; w_3 = 2\)).

> **Nota:** a formulação inicial incluía o custo interno como variável por alocação. Esta abordagem foi revista durante o projecto: o custo interno passou a ser tratado como custo fixo semanal comprometido, fora da função objetivo.

## 7. Nota sobre a evolução da formulação

A formulação apresentada neste notebook reflecte o modelo final implementado em `src/optimizer.py`. Ao longo do projecto, a formulação inicial foi refinada em vários aspectos:

- **Cenário híbrido:** a versão inicial considerava apenas colaboradores internos; a versão final incorpora recursos externos com variáveis \(y_{r,d,t,f}\) e activação condicional.
- **Custo interno fixo:** o tratamento do custo passou de variável por alocação para custo comprometido fora da função objetivo.
- **Elegibilidade funcional:** a qualificação mínima da formulação inicial foi generalizada para elegibilidade hierárquica, incorporando hierarquia funcional e exclusões individuais.
- **H4, H8, H9:** restrições presentes na formulação geral do problema; na implementação final, H4 ficou fora do modelo ILP ativo por exclusão do turno T3, enquanto H8 e H9 passaram a ser satisfeitas estruturalmente através dos conjuntos e parâmetros de entrada.
- **Pesos da função objetivo:** calibrados por análise de sensibilidade sobre 8 configurações; os pesos finais equilibram cobertura ideal, preferências e equidade de carga.